# Initial data quality review

This notebook checks the seven SAP exports. Source business values are preserved.

## 1. Load data

In [ ]:
from pathlib import Path
import hashlib
import re
from decimal import Decimal
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if isinstance(value, pd.DataFrame) else value)

DATA_DIR = None
if DATA_DIR is None:
    candidates = [p / relative
                  for p in [Path.cwd(), *Path.cwd().parents]
                  for relative in ['data/source', 'DIC_Arbeitsprobe_Testdaten_SAP_MM']]
    DATA_DIR = next((p for p in candidates if p.is_dir()), None)
if DATA_DIR is None or not Path(DATA_DIR).is_dir():
    raise FileNotFoundError('Set DATA_DIR to the directory containing the seven CSV files.')
DATA_DIR = Path(DATA_DIR)

spec = {
    'LFA1': ('LFA1_VENDOR_MASTER.csv', ['LIFNR']),
    'MARA': ('MARA_MATERIAL_MASTER.csv', ['MATNR']),
    'EKKO': ('EKKO_PO_HEADER.csv', ['MANDT', 'EBELN']),
    'EKPO': ('EKPO_PO_ITEM.csv', ['MANDT', 'EBELN', 'EBELP']),
    'EKET': ('EKET_SCHEDULE_LINES.csv', ['MANDT', 'EBELN', 'EBELP', 'ETENR']),
    'LIKP': ('LIKP_DELIVERY_HEADER.csv', ['MANDT', 'VBELN']),
    'LIPS': ('LIPS_DELIVERY_ITEM.csv', ['MANDT', 'VBELN', 'POSNR']),
}
raw, data, source_hashes = {}, {}, {}
for name, (filename, keys) in spec.items():
    path = DATA_DIR / filename
    source_hashes[filename] = hashlib.sha256(path.read_bytes()).hexdigest()
    frame = pd.read_csv(path, sep=';', encoding='utf-8-sig', dtype=str,
                        keep_default_na=False, na_filter=False, on_bad_lines='error')
    frame.insert(0, 'source_record', range(2, len(frame) + 2))
    raw[name] = frame
    data[name] = frame.assign(**{c: frame[c].str.strip()
                                for c in frame.columns if c != 'source_record'})

issues = {}
def show_issue(name, frame, limit=8):
    issues[name] = frame.copy()
    print(f'{name}: {len(frame)} records (up to {limit} shown)')
    display(frame.head(limit))


## 2. Table overview and sample records

In [ ]:
overview = pd.DataFrame([
    {'table': name, 'file': filename, 'rows': len(raw[name]),
     'columns': len(raw[name].columns) - 1, 'candidate_key': ', '.join(keys)}
    for name, (filename, keys) in spec.items()
])
display(overview)
for name, frame in raw.items():
    print(name)
    display(frame.head(3))


## 3. Missing values and whitespace

In [ ]:
missing_rows = []
for name, frame in data.items():
    keys = spec[name][1]
    for col in frame.columns.drop('source_record'):
        blank = frame[col].eq('')
        spaces = raw[name][col].ne(frame[col])
        missing_rows.append({'table': name, 'column': col, 'key_column': col in keys,
                             'blank_rows': int(blank.sum()),
                             'blank_pct': round(100 * blank.mean(), 2),
                             'outer_whitespace_rows': int(spaces.sum())})
    show_issue(name + '_missing_key', frame.loc[frame[keys].eq('').any(axis=1)])
missing_profile = pd.DataFrame(missing_rows)
display(missing_profile)


## 4. Duplicates and key conflicts

In [ ]:
duplicate_rows = []
for name, frame in data.items():
    keys = spec[name][1]
    fields = list(frame.columns.drop('source_record'))
    valid = frame.loc[~frame[keys].eq('').any(axis=1)]
    counts = valid.groupby(keys, dropna=False).size()
    variants = valid.drop_duplicates(fields).groupby(keys, dropna=False).size()
    conflict_keys = variants[variants > 1].reset_index()[keys]
    conflicts = valid.merge(conflict_keys, on=keys, how='inner', validate='many_to_one')
    exact = raw[name].duplicated(fields, keep=False)
    duplicate_rows.append({
        'table': name, 'exact_extra_rows_raw': int(raw[name].duplicated(fields).sum()),
        'duplicate_key_groups': int((counts > 1).sum()),
        'conflicting_key_groups': len(conflict_keys)})
    show_issue(name + '_exact_duplicates', raw[name].loc[exact])
    show_issue(name + '_key_conflicts', conflicts)
duplicate_profile = pd.DataFrame(duplicate_rows)
display(duplicate_profile)


## 5. Date values

In [ ]:
date_columns = {'EKKO': ['BEDAT', 'AEDAT'], 'EKET': ['EINDT'], 'LIKP': ['WADAT_IST']}
parsed_dates, date_rows = {}, []
for name, columns in date_columns.items():
    for col in columns:
        values = data[name][col]
        shape = values.str.fullmatch(r'\d{4}-\d{2}-\d{2}')
        parsed = pd.to_datetime(values.where(shape), format='%Y-%m-%d', errors='coerce')
        parsed_dates[name, col] = parsed
        bad = values.ne('') & parsed.isna()
        date_rows.append({'table': name, 'column': col, 'missing': int(values.eq('').sum()),
                          'invalid_nonblank': int(bad.sum()), 'min': parsed.min(), 'max': parsed.max()})
        show_issue(name + '_' + col + '_invalid_date', data[name].loc[bad])
date_profile = pd.DataFrame(date_rows)
display(date_profile)
changed_before_created = parsed_dates['EKKO', 'AEDAT'] < parsed_dates['EKKO', 'BEDAT']
show_issue('EKKO_change_before_order_date', data['EKKO'].loc[changed_before_created])


## 6. Quantities and amounts

In [ ]:
numeric_columns = {'EKPO': ['MENGE', 'NETPR', 'NETWR'],
                   'EKET': ['MENGE', 'WEMNG'], 'LIPS': ['LFIMG']}
numbers, numeric_rows = {}, []
for name, columns in numeric_columns.items():
    for col in columns:
        values = data[name][col]
        valid = values.str.fullmatch(r'[+-]?\d+(?:\.\d+)?')
        parsed = values.map(lambda value: Decimal(value) if re.fullmatch(r'[+-]?\d+(?:\.\d+)?', value) else None)
        numbers[name, col] = parsed
        negative = parsed.map(lambda value: value is not None and value < 0)
        zero = parsed.map(lambda value: value is not None and value == 0)
        invalid = values.ne('') & ~valid
        good = parsed.dropna()
        numeric_rows.append({'table': name, 'column': col, 'missing': int(values.eq('').sum()),
                             'invalid_nonblank': int(invalid.sum()), 'negative': int(negative.sum()),
                             'zero': int(zero.sum()), 'min': str(good.min()) if len(good) else None,
                             'max': str(good.max()) if len(good) else None})
        show_issue(name + '_' + col + '_invalid_number', data[name].loc[invalid])
        show_issue(name + '_' + col + '_negative', data[name].loc[negative])
numeric_profile = pd.DataFrame(numeric_rows)
display(numeric_profile)
over_received = pd.Series([a is not None and b is not None and a > b
                          for a, b in zip(numbers['EKET', 'WEMNG'], numbers['EKET', 'MENGE'])],
                         index=data['EKET'].index)
show_issue('EKET_received_above_scheduled_review', data['EKET'].loc[over_received])


## 7. Table relationships

In [ ]:
relations = [
    ('EKPO', 'EKKO', ['MANDT', 'EBELN']),
    ('EKET', 'EKPO', ['MANDT', 'EBELN', 'EBELP']),
    ('LIPS', 'EKPO', ['MANDT', 'EBELN', 'EBELP']),
    ('LIPS', 'LIKP', ['MANDT', 'VBELN']),
    ('EKKO', 'LFA1', ['LIFNR']), ('LIKP', 'LFA1', ['LIFNR']),
    ('EKPO', 'MARA', ['MATNR']), ('LIPS', 'MARA', ['MATNR']),
]
relation_rows = []
for child_name, parent_name, keys in relations:
    child, parent = data[child_name], data[parent_name]
    blank = child[keys].eq('').any(axis=1)
    parent_valid = parent.loc[~parent[keys].eq('').any(axis=1)]
    parent_counts = parent_valid.groupby(keys).size().rename('parent_rows').reset_index()
    joined = child.loc[~blank].merge(parent_counts, on=keys, how='left', validate='many_to_one')
    missing = joined.loc[joined['parent_rows'].isna()]
    ambiguous = joined.loc[joined['parent_rows'].gt(1)]
    label = child_name + '_to_' + parent_name
    relation_rows.append({'relationship': label, 'child_rows': len(child),
                          'blank_reference_rows': int(blank.sum()), 'missing_parent_rows': len(missing),
                          'ambiguous_parent_rows': len(ambiguous)})
    show_issue(label + '_missing_parent', missing)
    show_issue(label + '_ambiguous_parent', ambiguous)
relation_profile = pd.DataFrame(relation_rows)
display(relation_profile)


## 8. Units, materials and branches

In [ ]:
comparison_rows = []
def compare_fields(child_name, parent_name, keys, fields):
    child, parent = data[child_name], data[parent_name]
    valid_parent = parent.loc[~parent[keys].eq('').any(axis=1)]
    unique_parent = valid_parent.loc[~valid_parent.duplicated(keys, keep=False)]
    right = unique_parent[keys + fields].rename(columns={c: 'parent_' + c for c in fields})
    merged = child.loc[~child[keys].eq('').any(axis=1)].merge(
        right, on=keys, how='inner', validate='many_to_one')
    for col in fields:
        comparable = merged[col].ne('') & merged['parent_' + col].ne('')
        mismatch = comparable & merged[col].ne(merged['parent_' + col])
        label = f'{child_name}_to_{parent_name}_{col}'
        comparison_rows.append({'check': label, 'child_rows': len(child),
                                'compared_rows': int(comparable.sum()),
                                'not_compared_rows': len(child) - int(comparable.sum()),
                                'mismatch_rows': int(mismatch.sum())})
        show_issue(label + '_mismatch', merged.loc[mismatch])

compare_fields('EKPO', 'MARA', ['MATNR'], ['MEINS'])
compare_fields('LIPS', 'MARA', ['MATNR'], ['MEINS'])
compare_fields('LIPS', 'EKPO', ['MANDT', 'EBELN', 'EBELP'], ['MEINS', 'MATNR', 'WERKS'])
compare_fields('LIPS', 'LIKP', ['MANDT', 'VBELN'], ['WERKS'])
comparison_profile = pd.DataFrame(comparison_rows)
display(comparison_profile)
for name, col in [('EKPO', 'MEINS'), ('LIPS', 'MEINS'), ('EKPO', 'LOEKZ'),
                  ('EKPO', 'WERKS'), ('LIKP', 'WERKS'), ('LIPS', 'WERKS')]:
    print(name, col)
    display(data[name][col].value_counts(dropna=False).rename_axis(col).reset_index(name='rows'))


## 9. Join multiplication

In [ ]:
item_key = ['MANDT', 'EBELN', 'EBELP']
schedule_counts = data['EKET'].groupby(item_key).size().rename('schedule_rows')
delivery_counts = data['LIPS'].groupby(item_key).size().rename('delivery_rows')
fanout = schedule_counts.to_frame().join(delivery_counts, how='inner').reset_index()
fanout['naive_join_rows'] = fanout['schedule_rows'] * fanout['delivery_rows']
fanout = fanout.loc[(fanout['schedule_rows'] > 1) & (fanout['delivery_rows'] > 1)]
show_issue('schedule_delivery_many_to_many', fanout.sort_values('naive_join_rows', ascending=False))


## 10. Findings

In [ ]:
findings = pd.DataFrame([{'check': name, 'records': len(frame)}
                         for name, frame in issues.items() if len(frame)])
if len(findings):
    display(findings.sort_values(['records', 'check'], ascending=[False, True]).reset_index(drop=True))
else:
    print('No issues found.')


assert all(hashlib.sha256((DATA_DIR / filename).read_bytes()).hexdigest() == digest
           for filename, digest in source_hashes.items()), 'A source file changed during execution.'
print('Source files unchanged.')


## 11. Sample size and ranking thresholds

Compare 1, 5, 10 and 20 order items per supplier, month and scope. The final scheduled date determines the month.
Only months closed by `SAMPLE_AS_OF_DATE` are included.

**Local preview:** Count unique order items with unambiguous attribution. Remove identical source rows; do not arbitrarily resolve key conflicts.
Read both Silver date formats. This preview does not apply all Silver and Gold rules.
`candidate_items_preview` is **not the evaluated order item count**. Quarantine and further exclusions can reduce it.
Items without a reliable month due to conflicts or invalid dates are not included.

**Decision:** Use Gold `evaluated_order_item_count` for the final threshold decision.
A threshold of 10 or 20 does not guarantee statistical reliability. First inspect its effect on coverage.

In [ ]:
SAMPLE_AS_OF_DATE = pd.Timestamp('2026-09-30')
SAMPLE_THRESHOLDS = [1, 5, 10, 20]

def sample_unique(name):
    fields = list(raw[name].columns.drop('source_record'))
    frame = raw[name][fields].drop_duplicates().copy()
    for column in fields:
        frame[column] = frame[column].str.strip()
    keys = spec[name][1]
    return frame.loc[~frame[keys].eq('').any(axis=1)
                     & ~frame.duplicated(keys, keep=False)].copy()

def sample_date(values):
    iso = pd.to_datetime(values.where(values.str.fullmatch(r'\d{4}-\d{2}-\d{2}')),
                         format='%Y-%m-%d', errors='coerce')
    german = pd.to_datetime(values.where(values.str.fullmatch(r'\d{2}\.\d{2}\.\d{4}')),
                            format='%d.%m.%Y', errors='coerce')
    return iso.fillna(german)

sample_items = sample_unique('EKPO')
sample_orders = sample_unique('EKKO')
sample_schedules = sample_unique('EKET')
sample_schedules['due_date'] = sample_date(sample_schedules['EINDT'])
# Do not infer a month from incomplete schedule records.
sample_raw_schedules = data['EKET'].copy()
sample_raw_schedules['due_date'] = sample_date(sample_raw_schedules['EINDT'])
schedule_key = spec['EKET'][1]
sample_bad_schedule_keys = sample_raw_schedules.merge(
    sample_schedules[schedule_key].assign(_unique_schedule=True),
    on=schedule_key, how='left', validate='many_to_one')
sample_bad_items = sample_bad_schedule_keys.loc[
    sample_bad_schedule_keys['_unique_schedule'].isna()
    | sample_bad_schedule_keys['due_date'].isna(), item_key].drop_duplicates()
sample_due = sample_schedules.groupby(item_key, as_index=False)['due_date'].max()
sample_due = sample_due.merge(sample_bad_items.assign(_uncertain=True), on=item_key,
                              how='left', validate='one_to_one')
sample_due = sample_due.loc[sample_due['_uncertain'].isna()].drop(columns='_uncertain')
sample_candidates = (sample_items.loc[sample_items['LOEKZ'].eq('') & sample_items['WERKS'].ne('')]
    .merge(sample_orders.loc[sample_orders['LIFNR'].ne(''), ['MANDT', 'EBELN', 'LIFNR']],
           on=['MANDT', 'EBELN'], how='inner', validate='many_to_one')
    .merge(sample_due, on=item_key, how='inner', validate='one_to_one'))
sample_candidates['report_month'] = sample_candidates['due_date'].dt.to_period('M').dt.to_timestamp()
sample_candidates = sample_candidates.loc[
    sample_candidates['due_date'].notna()
    & ((sample_candidates['report_month'] + pd.offsets.MonthEnd(0)) <= SAMPLE_AS_OF_DATE)]
sample_scoped = pd.concat([
    sample_candidates.assign(scope='BRANCH'),
    sample_candidates.assign(scope='ALL', WERKS='ALL')], ignore_index=True)
sample_counts = sample_scoped.groupby(
    ['report_month', 'scope', 'WERKS', 'LIFNR']).size().reset_index(name='candidate_items_preview')
print('Local preview: candidates, not yet Gold evaluations.')
print(f'{len(sample_candidates)} attributable items; cutoff {SAMPLE_AS_OF_DATE.date()}')
display(sample_counts.sort_values(['scope', 'WERKS', 'report_month', 'candidate_items_preview', 'LIFNR']))

In [ ]:
def sample_comparison(frame, count_column):
    required = ['report_month', 'scope', 'WERKS', 'LIFNR', count_column]
    frame = frame[required].copy()
    frame['WERKS'] = frame['WERKS'].fillna('ALL')
    if frame.duplicated(['report_month', 'scope', 'WERKS', 'LIFNR']).any():
        raise ValueError('Multiple rows per supplier, month and scope.')
    frame[count_column] = pd.to_numeric(frame[count_column], errors='raise')
    if (frame[count_column].isna().any() or (frame[count_column] < 0).any()
            or (frame[count_column] % 1 != 0).any()):
        raise ValueError('Item counts must be non-negative integers.')
    rows = []
    for (month, scope, branch), group in frame.groupby(['report_month', 'scope', 'WERKS']):
        counts = group[count_column]
        for threshold in SAMPLE_THRESHOLDS:
            eligible = counts >= threshold
            rows.append(dict(report_month=month, scope=scope, WERKS=branch,
                min_items=threshold, suppliers_total=len(group),
                suppliers_with_1_or_2_items=int(counts.between(1, 2).sum()),
                median_items=float(counts.median()), min_observed=int(counts.min()),
                max_observed=int(counts.max()), eligible_suppliers=int(eligible.sum()),
                ranking_slots=min(3, int(eligible.sum())),
                retained_item_pct=round(100 * counts[eligible].sum() / counts.sum(), 1)
                    if counts.sum() else None))
    return pd.DataFrame(rows)

sample_threshold_preview = sample_comparison(sample_counts, 'candidate_items_preview')
print('Local candidate threshold comparison. Gold may contain fewer evaluable items.')
display(sample_threshold_preview)

### Cross-check against Gold

In Databricks, the next cell reads `supplier_performance_dev.dev_gold.dev_supplier_monthly`.
Locally, set `GOLD_SAMPLE_CSV` to a comma-separated export of that view with its English column names.
No final threshold is recommended without Gold data. This check neither writes business tables nor changes `min_items`.

`eligible_suppliers` counts remaining suppliers. `ranking_slots` shows whether three places are possible.
`retained_item_pct` is the share of items belonging to suppliers above the threshold.
Inspect ALL and BRANCH separately. Company-wide coverage does not guarantee branch coverage.
The Gold view enforces the current user's access, so branch employees only see their permitted branches.

In [ ]:
GOLD_SAMPLE_CSV = None  # Optional: Path('gold_supplier_monthly.csv')
GOLD_SAMPLE_TABLE = 'supplier_performance_dev.dev_gold.dev_supplier_monthly'
gold_sample = None
if GOLD_SAMPLE_CSV is not None:
    gold_sample = pd.read_csv(GOLD_SAMPLE_CSV, dtype={'branch_code': str, 'supplier_id': str})
elif 'spark' in globals():
    if spark.catalog.tableExists(GOLD_SAMPLE_TABLE):
        gold_sample = spark.table(GOLD_SAMPLE_TABLE).toPandas()
    else:
        print(f'Gold view is missing: {GOLD_SAMPLE_TABLE}. Run Gold first.')
else:
    print('No local Gold connection. Set GOLD_SAMPLE_CSV for the cross-check.')

if gold_sample is not None:
    gold_sample = gold_sample.rename(columns={
        'branch_code': 'WERKS', 'supplier_id': 'LIFNR', 'report_scope': 'scope',
        'evaluated_order_item_count': 'evaluated_items', 'excluded_order_item_count': 'excluded_items',
        'evaluation_coverage_rate': 'coverage_rate', 'performance_score': 'score'})
    gold_sample['report_month'] = pd.to_datetime(gold_sample['report_month'], errors='raise')
    if 'as_of_date' not in gold_sample.columns:
        raise ValueError('The Gold export must contain as_of_date.')
    gold_cutoffs = pd.to_datetime(gold_sample['as_of_date'], errors='raise')
    if gold_cutoffs.isna().any() or not gold_cutoffs.eq(SAMPLE_AS_OF_DATE).all():
        raise ValueError('Gold cutoff and SAMPLE_AS_OF_DATE must match.')
    if not gold_sample['scope'].isin(['ALL', 'BRANCH']).all():
        raise ValueError('Unknown report scope in Gold.')
    gold_sample = gold_sample.loc[
        (gold_sample['report_month'] + pd.offsets.MonthEnd(0)) <= SAMPLE_AS_OF_DATE]
    gold_threshold_comparison = sample_comparison(gold_sample, 'evaluated_items')
    print('Gold: evaluated order items. Use these counts to choose the threshold.')
    display(gold_sample[['report_month', 'scope', 'WERKS', 'LIFNR', 'evaluated_items',
                        'excluded_items', 'coverage_rate', 'score']].sort_values(
                            ['scope', 'WERKS', 'report_month', 'evaluated_items', 'LIFNR']))
    display(gold_threshold_comparison)

### Local preview result (2026-10-10)

Cutoff: 2026-09-30. These are candidate counts, not the final Gold evaluation.

| Scope | Supplier-month groups | Median items | Maximum | Groups with at least 5 items | Groups with at least 10 / 20 |
|---|---:|---:|---:|---:|---:|
| ALL | 187 | 2 | 8 | 10 | 0 / 0 |
| BRANCH | 337 | 1 | 4 | 0 | 0 / 0 |

At threshold 5, only one of 20 months can provide three suppliers at ALL scope. No branch supplier qualifies.
Thresholds 10 and 20 produce no ranking in this preview. The parameter remains unchanged pending the Gold check.
Small samples can be shown in the demo, with their limited reliability clearly stated.
A longer evaluation period is an alternative, but would be a separate analysis from the monthly report.

## 12. Suppliers across the full period

One purchase order can contain several items. The ranking threshold counts items, not order headers.
This overview uses unique, non-conflicting source keys; it is not a full Silver/Gold validation.
`orders` counts EKKO orders. `order_items` counts attributable EKPO items, including deleted items.
`nondeleted_items` includes only items with blank LOEKZ. It is not a Gold evaluation count.
Include all suppliers in the unique supplier master and any additional supplier IDs found in orders.

In [ ]:
period_orders = sample_unique('EKKO')
period_orders = period_orders.loc[period_orders['LIFNR'].ne('')]
period_items = sample_unique('EKPO').merge(
    period_orders[['MANDT', 'EBELN', 'LIFNR']], on=['MANDT', 'EBELN'],
    how='inner', validate='many_to_one')
period_summary = sample_unique('LFA1')[['LIFNR', 'NAME1']]
for series in [period_orders.groupby('LIFNR').size().rename('orders'),
               period_items.groupby('LIFNR').size().rename('order_items'),
               period_items.loc[period_items['LOEKZ'].eq('')].groupby('LIFNR').size().rename('nondeleted_items')]:
    period_summary = period_summary.merge(series, on='LIFNR', how='outer')
period_columns = ['orders', 'order_items', 'nondeleted_items']
period_summary[period_columns] = period_summary[period_columns].fillna(0).astype(int)
period_dates = sample_date(period_orders['BEDAT'])
print(f'Order dates: {period_dates.min().date()} to {period_dates.max().date()}')
print(f'Order headers without a valid date: {period_dates.isna().sum()}')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(period_summary.sort_values('LIFNR').reset_index(drop=True))
display(period_summary[period_columns].agg(['sum', 'median', 'max', 'min']))
